<a href="https://colab.research.google.com/github/Kamalll7/Final_Project_PCD-Kelompok_6/blob/main/notebook/01_Helmet_Detection_CHT_HOG_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys, subprocess

REQUIRED = ["opencv-python-headless", "scikit-image", "scikit-learn",
            "matplotlib", "pandas", "numpy", "pillow"]

def _ensure(pkgs):
    missing = []
    for p in pkgs:
        mod = {"opencv-python-headless": "cv2", "scikit-image": "skimage",
               "scikit-learn": "sklearn", "pillow": "PIL"}.get(p, p)
        try:
            __import__(mod)
        except ImportError:
            missing.append(p)
    if missing:
        print("Memasang:", missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)

_ensure(REQUIRED)

In [ ]:
import os, json, random, warnings, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, cohen_kappa_score,
                             roc_curve, auc, classification_report)

warnings.filterwarnings("ignore")

# ---- Reproducibility -------------------------------------------------------
SEED = 7
random.seed(SEED)
np.random.seed(SEED)

# ---- Tampilan plot ---------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 110,
    "font.size": 9,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
})

print("NumPy       :", np.__version__)
print("OpenCV      :", cv2.__version__)
import sklearn, skimage
print("scikit-learn:", sklearn.__version__)
print("scikit-image:", skimage.__version__)

In [ ]:
DATASET_SLUG = "aneesarom/rider-with-helmet-without-helmet-number-plate"
CLASS_NAMES  = ["with helmet", "without helmet", "rider", "number plate"]

def _looks_like_dataset(p: Path) -> bool:
    return (p / "train" / "images").is_dir() and (p / "train" / "labels").is_dir()

def resolve_dataset() -> Path:
    """Cari folder dataset: env var -> path lokal umum -> cache kagglehub -> unduh."""
    candidates = []
    env = os.environ.get("PCD_DATASET_DIR")
    if env:
        candidates.append(Path(env))
    candidates += [Path("./dataset"), Path("../dataset"), Path("/content/dataset"),
                   Path.home() / ".cache/kagglehub/datasets/aneesarom/rider-with-helmet-without-helmet-number-plate"]

    for base in candidates:
        if not base.exists():
            continue
        if _looks_like_dataset(base):
            return base
        # kagglehub menyimpan pada subfolder bernomor versi
        for sub in base.rglob("train"):
            if (sub / "images").is_dir() and (sub / "labels").is_dir():
                return sub.parent

    print("Dataset belum ditemukan secara lokal, mengunduh via kagglehub ...")
    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=False)
        import kagglehub
    root = Path(kagglehub.dataset_download(DATASET_SLUG))
    for sub in [root] + list(root.rglob("*")):
        if _looks_like_dataset(sub):
            return sub
    raise FileNotFoundError("Struktur dataset tidak dikenali setelah unduhan.")

DATA_ROOT = resolve_dataset()
print("Dataset root :", DATA_ROOT.resolve())
print("Train images :", len(list((DATA_ROOT / 'train' / 'images').glob('*'))))
print("Val images   :", len(list((DATA_ROOT / 'val' / 'images').glob('*'))))